# Sample Participant Submission — Loan Default Prediction
**Participant:** Participant_001  
**Milestone:** Milestone_1  
**Purpose:** Synthetic demonstration submission for testing the evaluation workflow.

> This is sample content only. Replace it with a real participant's work before grading.

## 1. Problem understanding
The objective is to explore a loan dataset and build a baseline model that predicts whether a borrower defaults. A false negative may be costly because a risky loan could be classified as safe. Therefore, recall for the default class should be considered alongside precision and ROC-AUC.

This notebook is illustrative. The dataset used below is generated synthetically; it is not a real lender dataset.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 800

income = rng.normal(65000, 18000, n).clip(15000, 180000)
loan_amount = rng.normal(18000, 9000, n).clip(1000, 60000)
credit_score = rng.normal(680, 55, n).clip(450, 850)
employment_years = rng.integers(0, 31, n)
debt_to_income = rng.uniform(0.05, 0.75, n)

# Synthetic default probability: this is for demonstration, not a real risk model.
logit = (
    -2.2
    + 2.4 * (debt_to_income - 0.35)
    + 0.000025 * (loan_amount - income * 0.25)
    - 0.006 * (credit_score - 650)
    - 0.025 * employment_years
)
prob_default = 1 / (1 + np.exp(-logit))
default = rng.binomial(1, prob_default)

df = pd.DataFrame({
    "annual_income": income.round(0),
    "loan_amount": loan_amount.round(0),
    "credit_score": credit_score.round(0),
    "employment_years": employment_years,
    "debt_to_income": debt_to_income.round(3),
    "default": default
})
# Introduce a few missing values to demonstrate quality checks.
df.loc[rng.choice(df.index, 12, replace=False), "credit_score"] = np.nan
df.head()

## 2. Data understanding and quality checks

In [ ]:
print("Shape:", df.shape)
display(df.dtypes.to_frame("data_type"))
display(df.isna().sum().to_frame("missing_values"))
print("Duplicate rows:", df.duplicated().sum())
display(df["default"].value_counts().rename("count").to_frame())
display(df.describe().T)

## 3. Exploratory data analysis

In [ ]:
import matplotlib.pyplot as plt

df["default"].value_counts().sort_index().plot(kind="bar")
plt.title("Target class counts (0 = non-default, 1 = default)")
plt.xlabel("Default")
plt.ylabel("Number of borrowers")
plt.tight_layout()
plt.show()

df[["annual_income", "loan_amount", "credit_score", "debt_to_income"]].hist(
    figsize=(10, 7), bins=20
)
plt.tight_layout()
plt.show()

display(df.groupby("default")[["annual_income", "loan_amount", "credit_score", "debt_to_income"]].mean().round(2))

## 4. Data cleaning and feature preparation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

X = df.drop(columns="default")
y = df["default"]

numeric_features = X.columns.tolist()
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train rows:", len(X_train), "Test rows:", len(X_test))
print("Target prevalence — train:", round(y_train.mean(), 3), "test:", round(y_test.mean(), 3))

## 5. Logistic regression baseline

In [ ]:
baseline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])
baseline.fit(X_train, y_train)

pred = baseline.predict(X_test)
prob = baseline.predict_proba(X_test)[:, 1]
print(classification_report(y_test, pred, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print("ROC-AUC:", round(roc_auc_score(y_test, prob), 3))

## 6. Findings and limitations
- The notebook checks data shape, data types, missingness, duplicates, target distribution, and summary statistics.
- Median imputation and standardization are fitted within a pipeline to reduce leakage risk.
- Stratified splitting helps preserve the target-class ratio between training and test sets.
- `class_weight="balanced"` is a baseline choice; it should be compared against an unweighted model and alternative thresholds.
- The synthetic data and generated relationship are not suitable for lending decisions. Results are only for testing the submission-evaluation workflow.
- Further work should include cross-validation, threshold selection based on business costs, calibration, fairness checks, and independent validation.